[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C66_Agentic_Evaluation_Course/04_reliability/04_reliability_stats.ipynb)

# 04 · 可靠性与统计（方差分解 / pass@k 与 pass^k / 预算分配 / 聚类自举 / 配对检验 / 胜者诅咒）

目标：把「跑几次够不够」「差 3 个点算不算差」这些每天都要回答的问题，变成**可以算的数**。

本 notebook 你会亲手实现：
1. **方差分解** —— 把总方差拆成任务间与任务内，看清楚钱该花在 N 还是 k 上
2. **pass@k 与 pass^k 的无偏估计器** —— 组合数公式，以及为什么不能"跑 k 次数一数"
3. **预算最优分配** —— 给定预算，解出最优的 (任务数 N, 重复数 k)
4. **聚类自举 vs 朴素自举** —— 用覆盖率实验证明朴素自举的区间窄了近一半
5. **McNemar 与配对自举** —— 同一批任务上比较两个 agent
6. **胜者诅咒模拟** —— 20 个能力完全相同的模型，榜首会虚高多少

> 心智模型：**先量 bias（判分器），再压 var（多跑），最后写区间（聚类自举）。
> 顺序反了，你会得到一个对错误的量的精确估计。**

## 1 · 方差分解：任务间 vs 任务内

In [ ]:
import math, itertools
from collections import defaultdict
import numpy as np

def simulate_runs(N, k, mu=0.4, sd_task=0.35, seed=0):
    """生成 N 个任务 × k 次重复的 0/1 结果。
    每个任务有自己的成功率 p_i ~ Beta(由 mu, sd_task 反推)，任务内是伯努利采样。"""
    rng = np.random.default_rng(seed)
    # 由均值与标准差反推 Beta 参数
    v = sd_task ** 2
    common = mu * (1 - mu) / v - 1
    a, b = max(mu * common, 0.05), max((1 - mu) * common, 0.05)
    p = rng.beta(a, b, size=N)
    X = (rng.random((N, k)) < p[:, None]).astype(float)
    return X, p

X, p_true = simulate_runs(N=400, k=8, seed=1)
task_means = X.mean(axis=1)
grand = X.mean()

# 方差分解：between = 任务真实难度的方差；within = 同一任务重复之间的方差
var_between = float(np.var(p_true, ddof=1))
var_within = float(np.mean(p_true * (1 - p_true)))
print(f'总体成功率        {grand:.3f}')
print(f'任务间方差 σ²_b   {var_between:.4f}')
print(f'任务内方差 σ²_w   {var_within:.4f}')
print(f'σ²_w / σ²_b       {var_within/var_between:.2f}')

rho = var_between / (var_between + var_within)          # 组内相关系数
print(f'组内相关 ρ        {rho:.3f}')
assert 0 < rho < 1
print('\n✅ ρ 就是「同一任务的两次重复有多像」。ρ 越高，重复的边际价值越低。')

In [ ]:
def var_of_mean(N, k, var_b, var_w):
    return var_b / N + var_w / (N * k)

print(f"{'k':>4}{'Var(mean)':>14}{'标准差':>12}{'相对 k=1 的降幅':>18}")
base = var_of_mean(400, 1, var_between, var_within)
for k in [1, 2, 5, 10, 50, 1000]:
    v = var_of_mean(400, k, var_between, var_within)
    print(f'{k:>4}{v:>14.6f}{math.sqrt(v):>12.4f}{1 - math.sqrt(v/base):>17.1%}')

floor = var_between / 400
print(f'\nk → ∞ 的方差下限: {floor:.6f}（标准差 {math.sqrt(floor):.4f}）')
assert var_of_mean(400, 1000, var_between, var_within) > floor
print('✅ 无论跑多少次重复，方差都降不到任务间方差以下——')
print('   而把 N 翻倍，两项一起减半。这就是「先扩任务集，再加重复」的定量依据。')

## 2 · pass@k 与 pass^k 的无偏估计

给定同一任务跑了 $n$ 次、成功 $c$ 次：

$$\text{pass@}k = 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}}, \qquad
\text{pass}^k = \frac{\binom{c}{k}}{\binom{n}{k}}$$

In [ ]:
def pass_at_k(n, c, k):
    """n 次采样中成功 c 次，随机抽 k 次「至少一次成功」的概率（无偏）。"""
    if n - c < k:
        return 1.0
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)

def pass_pow_k(n, c, k):
    """随机抽 k 次「全部成功」的概率（无偏）。"""
    if c < k:
        return 0.0
    return math.comb(c, k) / math.comb(n, k)

n = 20
print(f"{'c/n':>8}{'p̂':>8}", end='')
for k in [1, 2, 3, 5, 8]:
    print(f'{"@"+str(k):>9}', end='')
for k in [1, 2, 3, 5, 8]:
    print(f'{"^"+str(k):>9}', end='')
print()
for c in [4, 10, 14, 18, 20]:
    print(f'{c:>3}/{n:<4}{c/n:>8.2f}', end='')
    for k in [1, 2, 3, 5, 8]:
        print(f'{pass_at_k(n, c, k):>9.3f}', end='')
    for k in [1, 2, 3, 5, 8]:
        print(f'{pass_pow_k(n, c, k):>9.3f}', end='')
    print()

assert abs(pass_at_k(20, 14, 1) - 0.7) < 1e-12
assert abs(pass_pow_k(20, 14, 1) - 0.7) < 1e-12
assert pass_at_k(20, 14, 8) > 0.99 and pass_pow_k(20, 14, 8) < 0.06
print('\n✅ 看 c=14 那一行：pass@8 是 99.9%（"接近完美"），pass^8 是 5.7%（"基本不可用"）。')
print('   同一个 agent，两个数字都没撒谎——判断标准只有一条：结果被执行前有没有人挑一挑。')

In [ ]:
# 为什么要用无偏估计器：跟「跑 k 次数一数」比方差
def naive_pass_pow_k(p, k, n_trials, rng):
    """朴素做法：直接跑 k 次看是否全对，重复 n_trials 次取平均。"""
    return float(np.mean([(rng.random(k) < p).all() for _ in range(n_trials)]))

rng = np.random.default_rng(7)
P_TRUE, K, N_SAMPLES = 0.7, 3, 20
truth = P_TRUE ** K

naive_est, unbiased_est = [], []
for _ in range(2000):
    draws = rng.random(N_SAMPLES) < P_TRUE
    c = int(draws.sum())
    unbiased_est.append(pass_pow_k(N_SAMPLES, c, K))
    # 朴素：只用前 K 次采样，看是否全对；再用剩下的样本重复几组
    groups = N_SAMPLES // K
    naive_est.append(float(np.mean([draws[g*K:(g+1)*K].all() for g in range(groups)])))

print(f'真值 pass^{K} = {truth:.4f}')
print(f'无偏估计器  均值 {np.mean(unbiased_est):.4f}  标准差 {np.std(unbiased_est):.4f}')
print(f'朴素分组法  均值 {np.mean(naive_est):.4f}  标准差 {np.std(naive_est):.4f}')
assert abs(np.mean(unbiased_est) - truth) < 0.02
assert np.std(unbiased_est) < np.std(naive_est)
print('\n✅ 两者都无偏，但无偏估计器的方差明显更小——')
print('   因为它用上了「从 n 次里抽 k 次」的全部组合信息，而不是把样本切成互不重叠的几组。')

## 3 · 预算最优分配：N 和 k 该怎么切

在预算 $B = N(c_{task} + k \cdot c_{run})$ 下最小化 $\frac{\sigma_b^2}{N} + \frac{\sigma_w^2}{Nk}$。
理论解 $k^* = \sqrt{\frac{\sigma_w^2}{\sigma_b^2}\cdot\frac{c_{task}}{c_{run}}}$，下面用网格搜索验证。

In [ ]:
def optimal_k(var_b, var_w, c_task, c_run, budget, k_max=40):
    """网格搜索最优 (N, k)。N 由预算与 k 决定：N = budget / (c_task + k*c_run)。"""
    best = None
    for k in range(1, k_max + 1):
        N = budget / (c_task + k * c_run)
        if N < 20:                       # 任务数太少，估计不可靠，直接排除
            continue
        v = var_b / N + var_w / (N * k)
        if best is None or v < best[2]:
            best = (int(N), k, v)
    return best

def theoretical_k(var_b, var_w, c_task, c_run):
    return math.sqrt((var_w / var_b) * (c_task / c_run))

SCENARIOS = [
    ('合成任务集（任务几乎免费）', 0.05, 0.20, 1.0, 1.0),
    ('公开基准（任务贵、跑一次也贵）', 0.05, 0.20, 20.0, 1.0),
    ('自建高保真环境（任务极贵）', 0.05, 0.20, 200.0, 1.0),
]
BUDGET = 20000
print(f"{'场景':<32}{'最优 N':>8}{'最优 k':>8}{'理论 k*':>10}")
for name, vb, vw, ct, cr in SCENARIOS:
    N_opt, k_opt, v = optimal_k(vb, vw, ct, cr, BUDGET)
    print(f'{name:<32}{N_opt:>8}{k_opt:>8}{theoretical_k(vb, vw, ct, cr):>10.1f}')

_, k_cheap, _ = optimal_k(0.05, 0.20, 1.0, 1.0, BUDGET)
_, k_expensive, _ = optimal_k(0.05, 0.20, 200.0, 1.0, BUDGET)
assert k_expensive > k_cheap
print('\n✅ 任务越贵，最优重复次数越高——因为「多跑一次」相对「多写一道题」变便宜了。')
print('   这解释了为什么 SWE-bench 类基准常见 --epochs 5，而合成任务集应该直接扩任务数。')

## 4 · 聚类自举 vs 朴素自举：覆盖率实验

构造一个已知真值的场景，两种自举各算 95% 区间，统计真值落在区间内的比例。
名义覆盖率应当是 95%——朴素自举会显著低于它。

In [ ]:
def naive_bootstrap_ci(X, n_boot=400, seed=0):
    """错误做法：把 N*k 条 rollout 当独立样本重采样。"""
    rng = np.random.default_rng(seed)
    flat = X.ravel()
    n = flat.size
    boots = [flat[rng.integers(0, n, n)].mean() for _ in range(n_boot)]
    return np.percentile(boots, [2.5, 97.5])

def clustered_bootstrap_ci(X, n_boot=400, seed=0):
    """正确做法：重采样「任务」，被抽中的任务连同它全部 k 次重复一起进来。"""
    rng = np.random.default_rng(seed)
    N = X.shape[0]
    boots = [X[rng.integers(0, N, N)].mean() for _ in range(n_boot)]
    return np.percentile(boots, [2.5, 97.5])

def coverage_experiment(n_rep=120, N=150, k=6, mu=0.4, sd_task=0.35):
    hit_naive = hit_clust = 0
    w_naive, w_clust = [], []
    for r in range(n_rep):
        X, p = simulate_runs(N, k, mu=mu, sd_task=sd_task, seed=1000 + r)
        truth = mu                                 # 推断目标：这类任务上的总体成功率
        lo1, hi1 = naive_bootstrap_ci(X, n_boot=300, seed=r)
        lo2, hi2 = clustered_bootstrap_ci(X, n_boot=300, seed=r)
        hit_naive += (lo1 <= truth <= hi1)
        hit_clust += (lo2 <= truth <= hi2)
        w_naive.append(hi1 - lo1)
        w_clust.append(hi2 - lo2)
    return (hit_naive / n_rep, hit_clust / n_rep, np.mean(w_naive), np.mean(w_clust))

cn, cc, wn, wc = coverage_experiment()
print(f'名义覆盖率 95%')
print(f'  朴素自举  实际覆盖 {cn:.1%}   平均区间宽度 {wn:.4f}')
print(f'  聚类自举  实际覆盖 {cc:.1%}   平均区间宽度 {wc:.4f}')
print(f'  区间宽度比: 聚类 / 朴素 = {wc/wn:.2f}x')
assert cc > cn, '聚类自举的覆盖率应当更接近名义值'
assert wc > wn, '聚类自举的区间必然更宽'
print('\n✅ 朴素自举的区间窄了一大截，覆盖率因此低于名义的 95%——')
print('   这就是评测报告里「误差棒画得太窄」的直接来源。')

In [ ]:
# 有效样本量：n_eff = Nk / (1 + (k-1)ρ)
def n_eff(N, k, rho):
    return N * k / (1 + (k - 1) * rho)

print(f"{'ρ':>6}{'名义 Nk':>10}{'n_eff':>10}{'占比':>8}")
for rho_ in [0.0, 0.3, 0.5, 0.7, 0.9]:
    ne = n_eff(500, 5, rho_)
    print(f'{rho_:>6.1f}{2500:>10}{ne:>10.0f}{ne/2500:>8.0%}')

assert abs(n_eff(500, 5, 0.0) - 2500) < 1e-9
assert n_eff(500, 5, 1.0) == 500
print('\n✅ ρ=0 时 n_eff = Nk（重复完全独立）；ρ=1 时 n_eff = N（重复毫无新信息）。')
print('   agent 评测的 ρ 通常在 0.5-0.8，所以名义样本量要打三折左右看。')

## 5 · 配对检验：McNemar 与配对自举

In [ ]:
def mcnemar(x, y, continuity=True):
    """x, y: 同一批任务上两个 agent 的 0/1 结果。返回 (b, c, chi2, p 近似)。
    b = x 对 y 错的任务数; c = x 错 y 对的任务数。一致对不携带信息。"""
    x, y = np.asarray(x, dtype=bool), np.asarray(y, dtype=bool)
    b = int((x & ~y).sum())
    c = int((~x & y).sum())
    if b + c == 0:
        return b, c, 0.0, 1.0
    num = abs(b - c) - (1 if continuity else 0)
    chi2 = max(num, 0) ** 2 / (b + c)
    # 卡方(1) 的上尾概率：p = erfc(sqrt(chi2/2))
    p = math.erfc(math.sqrt(chi2 / 2))
    return b, c, chi2, p

def paired_bootstrap(x, y, n_boot=4000, seed=0):
    rng = np.random.default_rng(seed)
    d = np.asarray(x, dtype=float) - np.asarray(y, dtype=float)
    n = len(d)
    boots = [d[rng.integers(0, n, n)].mean() for _ in range(n_boot)]
    return float(d.mean()), tuple(np.percentile(boots, [2.5, 97.5]))

rng = np.random.default_rng(5)
N = 500
task_diff = rng.beta(2, 3, size=N)                 # 任务难度（两个 agent 共享）
A = (rng.random(N) < task_diff + 0.03).astype(int)  # A 真实略强 3 个点
B = (rng.random(N) < task_diff).astype(int)

b, c, chi2, p = mcnemar(A, B)
diff, (lo, hi) = paired_bootstrap(A, B, seed=2)
print(f'A 成功率 {A.mean():.1%} | B 成功率 {B.mean():.1%}')
print(f'不一致对: b={b} (A对B错), c={c} (A错B对) | 一致对 {N-b-c} 条完全不携带信息')
print(f'McNemar chi2={chi2:.2f}, p={p:.3f}')
print(f'配对自举: 差值 {diff:+.1%}  95% CI [{lo:+.1%}, {hi:+.1%}]')
assert b + c < N, '一致对占了大多数——这正是配对设计省样本的原因'
print(f'\n✅ {N} 道任务里只有 {b+c} 条携带信息（{(b+c)/N:.0%}）。')
print('   配对设计的全部威力，就是把「估计两个绝对值」变成「只看不一致对」。')

In [ ]:
# 独立设计 vs 配对设计的样本量对比
def required_n_independent(p1, p2, alpha=0.05, power=0.8):
    z_a, z_b = 1.959963985, 0.8416212336
    pb = (p1 + p2) / 2
    num = (z_a * math.sqrt(2 * pb * (1 - pb)) + z_b * math.sqrt(p1*(1-p1) + p2*(1-p2))) ** 2
    return math.ceil(num / (p1 - p2) ** 2)

def required_n_paired(p_disc, delta, alpha=0.05, power=0.8):
    z_a, z_b = 1.959963985, 0.8416212336
    return math.ceil(((z_a + z_b) ** 2 * p_disc) / (delta ** 2))

p_disc = (b + c) / N
print(f"{'要检出的差异':>14}{'独立设计':>12}{'配对设计':>12}{'节省':>10}")
for delta in [0.10, 0.05, 0.03]:
    ni = required_n_independent(0.40, 0.40 + delta)
    npd = required_n_paired(p_disc, delta)
    print(f'{delta:>14.0%}{ni:>12,}{npd:>12,}{ni/npd:>9.1f}x')

mde_paired = math.sqrt(((1.959963985 + 0.8416212336) ** 2 * p_disc) / N)
print(f'\n本次评测（N={N}, 不一致率 {p_disc:.0%}）的最小可检测差异 MDE = {mde_paired:.1%}')
assert mde_paired > 0
print(f'实测差异 {diff:+.1%} —— 是否超过 MDE: {abs(diff) > mde_paired}')
print('✅ 这一行就是报告里必须写的那句话：「本次评测能检出 ≥X 个点的差异」。')
print('   没有它，「不显著」这三个字没有任何信息量。')

## 6 · 胜者诅咒：20 个能力完全相同的模型，榜首虚高多少

In [ ]:
def winner_curse(m_models=20, N=500, true_p=0.40, n_rep=2000, seed=0):
    """所有模型真实能力完全相同，看观测榜首的平均分数与真值的差距。"""
    rng = np.random.default_rng(seed)
    gaps, reversal = [], 0
    for _ in range(n_rep):
        obs = rng.binomial(N, true_p, size=m_models) / N
        winner = int(np.argmax(obs))
        gaps.append(obs.max() - true_p)
        # 复现实验：换一批任务重跑，原榜首还是第一吗
        obs2 = rng.binomial(N, true_p, size=m_models) / N
        reversal += (int(np.argmax(obs2)) != winner)
    return float(np.mean(gaps)), reversal / n_rep

sigma = math.sqrt(0.40 * 0.60 / 500)
for m in [2, 5, 20, 100]:
    gap, rev = winner_curse(m_models=m)
    theory = sigma * math.sqrt(2 * math.log(m)) if m > 1 else 0.0
    print(f'm={m:>4} 模型 | 榜首平均虚高 {gap:+.2%} (理论 ≈ {theory:+.2%}) | 复现时榜首易主 {rev:.0%}')

gap20, rev20 = winner_curse(m_models=20)
gap2, _ = winner_curse(m_models=2)
assert gap20 > gap2, '参赛者越多，榜首虚高越严重'
assert rev20 > 0.8, '能力相同时，复现实验的榜首几乎必然易主'
print(f'\n✅ 20 个能力完全相同的模型，榜首平均虚高 {gap20:.1%}，')
print(f'   而且复现时有 {rev20:.0%} 的概率换人当第一——没有任何人作弊。')
print('   读榜单的经验法则：500 道任务的基准，相差 4 个点以内的名次本质上是并列。')

## ✏️ 练习 1：pass^k 的一致性缺口

实现 `consistency_gap(n, c, k)`：返回 `pass_pow_k(n,c,1) - pass_pow_k(n,c,k)`，
即「能力」与「可靠性」的分离量。行为完全确定的 agent 这个值应为 0。

In [ ]:
def consistency_gap(n, c, k):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(consistency_gap(20, 20, 5)) < 1e-12      # 20/20 全对 → 完全确定 → 缺口 0
assert abs(consistency_gap(20, 0, 5)) < 1e-12       # 0/20 全错 → 也完全确定 → 缺口 0
g = consistency_gap(20, 10, 5)
assert g > 0.4                                       # 一半一半 → 最不稳定
for c in [2, 6, 10, 14, 18]:
    print(f'c={c:>3}/20  pass^1={pass_pow_k(20,c,1):.2f}  pass^5={pass_pow_k(20,c,5):.3f}  '
          f'缺口={consistency_gap(20,c,5):.3f}')
print('✅ 练习 1 通过：缺口在 c/n≈0.5 附近最大——')
print('   能力决定 pass^k 曲线的起点，一致性决定它下降的斜率。')

## ✏️ 练习 2：给定预算求最小可检测差异

实现 `mde_paired(N, p_discordant, alpha=0.05, power=0.8)`：
由 `required_n_paired` 的公式反解，返回在 N 道任务上能检出的最小差异
$\delta = \sqrt{\frac{(z_\alpha + z_\beta)^2 p_{disc}}{N}}$。

In [ ]:
def mde_paired(N, p_discordant, alpha=0.05, power=0.8):
    # TODO（alpha/power 固定用 z_a=1.959963985, z_b=0.8416212336）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
m1 = mde_paired(500, 0.20)
assert required_n_paired(0.20, m1) <= 501            # 反解自洽
assert mde_paired(2000, 0.20) < m1                   # 任务多 → 能检出更小的差异
assert mde_paired(500, 0.40) > m1                    # 不一致率高 → 噪声大 → MDE 变大
for N_ in [200, 500, 1000, 4000]:
    print(f'N={N_:>5}  不一致率20%  MDE = {mde_paired(N_, 0.20):.2%}')
print('✅ 练习 2 通过：这一行数字应当在你决定「跑多少」之前就算出来——')
print('   而不是等实验跑完发现「不显著」再回头算。')

## ✏️ 练习 3：有效样本量与「该不该再加重复」

实现 `marginal_value_of_k(N, k, rho)`：返回把重复数从 `k` 加到 `k+1` 带来的
有效样本量增量占比，即 `(n_eff(N,k+1,rho) - n_eff(N,k,rho)) / n_eff(N,k,rho)`。

In [ ]:
def marginal_value_of_k(N, k, rho):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
v1 = marginal_value_of_k(500, 1, 0.7)
v5 = marginal_value_of_k(500, 5, 0.7)
assert v1 > v5 > 0, '边际收益必须递减且为正'
assert marginal_value_of_k(500, 1, 0.0) > marginal_value_of_k(500, 1, 0.9)
print(f"{'k→k+1':>8}{'ρ=0.3':>10}{'ρ=0.7':>10}{'ρ=0.9':>10}")
for k in [1, 2, 5, 10]:
    print(f'{f"{k}→{k+1}":>8}', end='')
    for r in [0.3, 0.7, 0.9]:
        print(f'{marginal_value_of_k(500, k, r):>10.1%}', end='')
    print()
print('✅ 练习 3 通过：ρ=0.7 时从 5 次加到 6 次只买到几个百分点的有效样本量——')
print('   这个数字就是「该停手了」的信号。')

## ✏️ 练习 4：把报告模板写成代码

实现 `eval_report(X, X_base=None, rho=None)`：输入 N×k 的 0/1 结果矩阵，
返回一个字典，含 `pass1`、`ci`（聚类自举 95% 区间）、`n_eff`、
`pass_at_k`、`pass_pow_k`（都取 k = 矩阵的列数）、以及在给了 `X_base` 时的
`paired_diff` 与 `mde`。ρ 未给时用 `var_between/(var_between+var_within)` 从数据估计。

In [ ]:
def eval_report(X, X_base=None, rho=None):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
Xa, _ = simulate_runs(N=300, k=5, mu=0.42, seed=11)
Xb, _ = simulate_runs(N=300, k=5, mu=0.38, seed=12)
rep = eval_report(Xa, Xb)
need = {'pass1', 'ci', 'n_eff', 'pass_at_k', 'pass_pow_k', 'paired_diff', 'mde'}
assert need <= set(rep)
assert rep['ci'][0] <= rep['pass1'] <= rep['ci'][1]
assert rep['n_eff'] <= 300 * 5
assert rep['pass_at_k'] >= rep['pass1'] >= rep['pass_pow_k']
for k_, v in rep.items():
    print(f'  {k_:<14} {v}')
print('\n✅ 练习 4 通过：这个字典直接对应讲解第 7 节的报告模板——')
print('   把它接进你的评测流水线，报告就再也不会漏掉 n_eff 和 MDE 这两行。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def consistency_gap(n, c, k):
    return pass_pow_k(n, c, 1) - pass_pow_k(n, c, k)

In [ ]:
# 练习 2 参考答案
def mde_paired(N, p_discordant, alpha=0.05, power=0.8):
    z_a, z_b = 1.959963985, 0.8416212336
    return math.sqrt(((z_a + z_b) ** 2 * p_discordant) / N)

In [ ]:
# 练习 3 参考答案
def marginal_value_of_k(N, k, rho):
    a = n_eff(N, k, rho)
    b = n_eff(N, k + 1, rho)
    return (b - a) / a

In [ ]:
# 练习 4 参考答案
def eval_report(X, X_base=None, rho=None):
    X = np.asarray(X, dtype=float)
    N, k = X.shape
    task_means = X.mean(axis=1)
    if rho is None:
        vb = float(np.var(task_means, ddof=1))
        vw = float(np.mean(task_means * (1 - task_means)))
        rho = vb / (vb + vw) if (vb + vw) > 0 else 0.0
    lo, hi = clustered_bootstrap_ci(X, seed=0)
    c_total = int(X.sum())
    out = {
        'pass1': round(float(X.mean()), 4),
        'ci': (round(float(lo), 4), round(float(hi), 4)),
        'n_eff': round(n_eff(N, k, rho), 1),
        'pass_at_k': round(float(np.mean([pass_at_k(k, int(r.sum()), k) for r in X])), 4),
        'pass_pow_k': round(float(np.mean([pass_pow_k(k, int(r.sum()), k) for r in X])), 4),
    }
    if X_base is not None:
        B = np.asarray(X_base, dtype=float)
        a1 = (X.mean(axis=1) > 0.5).astype(int)
        b1 = (B.mean(axis=1) > 0.5).astype(int)
        bb, cc_, _, _ = mcnemar(a1, b1)
        p_disc = (bb + cc_) / len(a1)
        out['paired_diff'] = round(float(X.mean() - B.mean()), 4)
        out['mde'] = round(mde_paired(len(a1), max(p_disc, 1e-6)), 4)
    return out

---
## 🧪 真实工程胶囊：把统计接进评测流水线

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. 跑评测时必须落盘的三个字段（否则统计全做不了）
# ══════════════════════════════════════════════════════════════════
# task_id, attempt, score  —— 三列就够了，其余都能从它们算出来
# inspect-ai: --epochs 5 会自动为每个样本产出 5 条带 epoch 编号的记录
#   inspect eval mytask.py --model anthropic/claude-sonnet-5 --epochs 5
#   inspect view    # 结果里每条 sample 都有 epoch 字段 = 我们的 attempt

# ══════════════════════════════════════════════════════════════════
# B. 从结果文件到报告的一段脚本（照抄改路径即可）
# ══════════════════════════════════════════════════════════════════
import pandas as pd, numpy as np
df = pd.read_json("results.jsonl", lines=True)          # task_id, attempt, score
piv = df.pivot_table(index="task_id", columns="attempt", values="score")
X = piv.to_numpy()                                       # N × k 矩阵，正是本 notebook 的输入
rep = eval_report(X)
print(rep)

# ══════════════════════════════════════════════════════════════════
# C. 两个模型对比：必须同一次运行、同一 harness、同一任务集
# ══════════════════════════════════════════════════════════════════
# 错误做法：上周跑 A，这周跑 B（中间镜像更新过 → 不是配对设计）
# 正确做法：
#   for model in [A, B]:
#       run_eval(model, tasks=TASKS_V12, harness="harness@1.7.2", epochs=5, seed=0)
#   然后用 mcnemar(a_scores, b_scores) + paired_bootstrap 出结论

# ══════════════════════════════════════════════════════════════════
# D. 报告里必须有的六行（缺一行就会被误读）
# ══════════════════════════════════════════════════════════════════
# 1. 指标全名        pass^1 (micro)，不是"成功率"
# 2. N 和 k 分开写   N=500 tasks × k=5 attempts，不是"2500 次运行"
# 3. 区间与方法      [30.1%, 38.4%] 按任务聚类自举, 2000 次
# 4. n_eff 与 ρ      658 (ρ=0.70)
# 5. MDE             8.1 个百分点 (α=0.05, power=0.8, 配对)
# 6. harness 指纹    harness@1.7.2, image sha256:..., concurrency=8
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 用在哪 |
|---|---|---|
| 四个方差源 | 任务间 / 任务内可用统计处理；模拟器与 harness 必须用工程消除 | 设计实验 |
| $k$ 有天花板，$N$ 没有 | 重复只能把方差压到 $\sigma_b^2/N$；先扩任务集再加重复 | 预算分配 |
| pass@k vs pass^k | 结果被执行前有没有人挑一挑，决定用哪个 | 选指标 |
| 配对设计 | 只有不一致对携带信息，样本量省一半以上 | 模型对比 |
| 聚类自举 | 独立单位是任务不是 rollout；朴素自举区间窄近一半 | 算区间 |
| MDE | 「不显著」必须配着检测下限一起报 | 写结论 |
| 胜者诅咒 | 20 个同水平模型，榜首虚高约 5 个点且复现必易主 | 读榜单 |

下一模块：**05 · 成本感知评测与 harness 可复现性**——
把「谁更强」这个问题改写成「给定预算谁更强」，并把 harness 钉死到可以被别人复现。